# Quadro ArtCraft · ComfyUI no Kaggle (vídeo grátis)

**Antes de rodar (uma vez):**
1. No painel direito, ligue **Internet** (Settings › Internet on) e escolha o acelerador **GPU T4 x2** (ou P100).
2. Sua conta do Kaggle precisa estar com o **telefone verificado**.
3. Clique em **Run All**. Leva uns 10 a 15 minutos na primeira vez (baixa ~18 GB de modelo). Deixe esta aba aberta.
4. No fim aparece um endereço `https://….trycloudflare.com`. Copie e cole em **⚙ › ComfyUI** no quadro.

Cada vez que a sessão do Kaggle acabar (até 12 h), rode de novo e cole o endereço novo.

In [ ]:
import os, subprocess, sys, time, json, re, urllib.request
BASE = "/tmp/ComfyUI"
if not os.path.exists(BASE):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", BASE], check=True)
# o Kaggle já traz o PyTorch com GPU: não reinstala
reqs = [l for l in open(f"{BASE}/requirements.txt").read().splitlines() if not re.match(r"^(torch|torchvision|torchaudio)\b", l.strip())]
open("/tmp/req_sem_torch.txt", "w").write("\n".join(reqs))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req_sem_torch.txt"], check=True)
print("ComfyUI instalado")

In [ ]:
HF = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"
ARQ = [
    (f"{HF}/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", f"{BASE}/models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors"),
    (f"{HF}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", f"{BASE}/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
    (f"{HF}/vae/wan2.2_vae.safetensors", f"{BASE}/models/vae/wan2.2_vae.safetensors"),
]
for url, destino in ARQ:
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    if os.path.exists(destino) and os.path.getsize(destino) > 10_000_000:
        print("já tem", os.path.basename(destino)); continue
    print("baixando", os.path.basename(destino), "…")
    subprocess.run(["wget", "-q", "-c", "-O", destino, url], check=True)
print("modelos prontos")

In [ ]:
subprocess.run(["wget", "-q", "-O", "/tmp/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
os.chmod("/tmp/cloudflared", 0o755)
comfy = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"], cwd=BASE, stdout=open("/tmp/comfy.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); break
    except Exception:
        time.sleep(2)
else:
    print(open("/tmp/comfy.log").read()[-3000:]); raise SystemExit("o ComfyUI não subiu")
info = json.load(urllib.request.urlopen("http://127.0.0.1:8188/system_stats"))
print("ComfyUI ligado:", [d.get("name") for d in info.get("devices", [])])

In [ ]:
# confere se a receita pronta do quadro combina com este ComfyUI
nos = json.load(urllib.request.urlopen("http://127.0.0.1:8188/object_info"))
faltam = [n for n in ["UNETLoader","CLIPLoader","VAELoader","ModelSamplingSD3","CLIPTextEncode","LoadImage","Wan22ImageToVideoLatent","KSampler","VAEDecode","CreateVideo","SaveVideo"] if n not in nos]
print("Receita: tudo certo" if not faltam else f"ATENÇÃO: este ComfyUI não tem os nós {faltam}")

In [ ]:
tunel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
endereco = None
t0 = time.time()
while time.time() - t0 < 90 and not endereco:
    linha = tunel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linha or "")
    if m: endereco = m.group(0)
print("=" * 70)
print("COLE ESTE ENDEREÇO NO QUADRO (⚙ › ComfyUI):")
print(endereco or "não consegui abrir o endereço; rode esta célula de novo")
print("=" * 70)
print("Deixe esta aba aberta. Quem tiver o endereço usa a sua GPU: não compartilhe.")

In [ ]:
# mantém a sessão viva enquanto você gera no quadro
while True:
    time.sleep(60)
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=5)
    except Exception:
        print("o ComfyUI parou; rode a célula de início de novo"); break